In [ ]:
import torch
import sys
import numpy as np
from ttpi import TTPI
from dynamic_systems import SinglePendulum
torch.set_default_dtype(torch.float64)
from plot_utils import plt_pendulum
%load_ext autoreload
%autoreload 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:

dt=0.01

state_max = torch.tensor([torch.pi,4*torch.pi]).to(device) # (theta,dtheta)
state_min = -1*state_max

n_state = [100]*2
n_action = 200
mass = 1.0; length=1.0; g= 9.81; coef_viscous = 0.01
action_max = 0.25*mass*g*length # 10% of inertial
action_min = -1*action_max

In [ ]:
domain_state = []
for i in range(2):#x,dx
    x_n = torch.linspace(state_min[i],0,int(n_state[i]/2)).to(device)
    x_p = torch.linspace(0,state_max[i],int(n_state[i]/2)).to(device)[1:]
    domain_state.append(torch.concat((x_n,x_p),dim=-1))


In [ ]:
from tt_utils import get_exponential_discretization

domain_action_pos = torch.linspace(action_min,0,int(n_action/2)).to(device)
domain_action_neg = -1*domain_action_pos
domain_action = [torch.concat((domain_action_neg ,domain_action_pos),dim=-1)]


In [ ]:
w_goal=1.0
w_action=0.1
w_scale = 1
a_max = torch.tensor([action_max]).to(device).view(-1,1)
a_min = -1*a_max
sys = SinglePendulum(mass=mass,coef_viscous = coef_viscous, 
                     length=length, state_min=state_min, 
                     state_max=state_max, action_max=a_max, action_min=a_min, 
                     dt=dt, w_scale=w_scale, w_goal=w_goal, w_action=w_action, device=device)


In [ ]:
def forward_model(state,action):
    return sys.forward_simulate(state,action)

def reward(state,action):
    rewards = sys.reward_state_action(state,action)
    return rewards

In [ ]:
sys_test = SinglePendulum(dt=dt, mass=1.*mass,coef_viscous=1*coef_viscous, length=1.*length, state_min=state_min, state_max=state_max, action_max=action_max, action_min=action_min, w_scale=w_scale, w_goal=w_goal, w_action=w_action, device=device)
def callback(ttdp, T=10, sys_t=sys, animation=False, file_name='file', callback_count=0):
    print("Testing....")
    state = torch.tensor([[0.,0.],
                        [0,0.1],
                        [0,0.2],
                        [0.1,-0.1],
                        [-0.5,0],
                        [0.2,0.2],
                        [-0.2,0.3]]).to(device).view(-1,2)
    history = []
    
    dt = sys_t.dt
    T=int(T/dt)
    traj = torch.zeros(state.shape[0],T,2)
    cum_reward = torch.tensor([0.]*state.shape[0]).to(device)
    action_history = torch.zeros(state.shape[0],T,1) #batch_size x T x 1
    for i in range(T):
        action = ttdp.policy(state) # batch_size x 1
        action_history[:,i,:] = action.clone().cpu()
        r = ttdp.reward_normalized(state,action)#reward_test(state,action)
        cum_reward+=r#reward_test(state,action)
        state = sys_t.forward_simulate(state,action)
        traj[:,i,:] = state.clone().cpu()
    print("Avg cumulative reward: ", torch.mean(cum_reward))
    theta_t = traj[0,:,0].cpu()
    dtheta_t = traj[0,:,1].cpu()

    from matplotlib import pyplot as plt0
    # plt0.plot(theta_t,dtheta_t)
    # plt0.xlim([state_min[0].cpu()-0.1,state_max[0].cpu()+0.1])
    # plt0.ylim([state_min[1].cpu()-0.1,state_max[1].cpu()+0.1])
    # plt0.grid()
    # plt0.ylabel(r'Velocity, rad/s')
    # plt0.xlabel('Angle, rad',fontsize=10)
    # plt0.title('Angle Vs Velocity')
    # plt0.show()
    
    # plt0.plot(np.arange(len(traj[0,:,1]))*dt, np.pi-np.abs(traj[0,:,1].cpu()))
    # plt0.grid()
    # plt0.ylabel(r'Angle, rad')
    # plt0.xlabel('Time in seconds',fontsize=10)
    # plt0.title('Angle')
    # plt0.show()
    plt0.plot(np.arange(len(traj[0,:,1]))*dt,traj[0,:,1].cpu())
    plt0.grid()
    plt0.ylabel(r'Joint velocity, rad/s')
    plt0.xlabel('Time in seconds',fontsize=10)
    plt0.title('velocity')
    plt0.show()
    # plt0.plot(action_history[0,:,0])
    # plt0.title('torque')
    # plt0.grid()
    # plt0.show()

    plt=plt_pendulum(theta_t.to('cpu').numpy(), 
                    figsize=5, dt=dt, scale=10, skip=50, animation=animation)
    plt.show()
    total_reward = torch.mean(cum_reward)
    return r.mean().to('cpu'), total_reward.to('cpu')
    

In [ ]:
ttpi = TTPI(domain_state=domain_state, domain_action=domain_action, reward=reward, 
                forward_model=forward_model, gamma=0.999, n_steps=1,
                rmax_v=100, rmax_a=100, nswp_v=10, nswp_a=10, 
                kickrank_v=10, kickrank_a=10,
                max_batch_v=10**4,max_batch_a=10**5,
                eps_cross_v=1e-3, 
                eps_cross_a=1e-3,
                eps_round_v=1e-3, 
                eps_round_a=1e-3, 
                n_samples=1, normalize_reward=False,
                verbose=True,
                device=device) # action = 'deterministic_tt', 'stochastic_tt', 'random'

In [ ]:

resume=False
ttpi.train(n_iter_max=1000,resume=resume, 
        callback=callback, callback_freq=25,
        verbose=False, file_name='swingup')
